# NB 63 — man / man−woman / woman steering: pooled vs hierarchical vs frame

Three guides (**man**, **man−woman**, **woman**) × three methods (**pooled**,
**hierarchical** subspace, **frame**/FRH) × **λ ∈ {0, 0.4}**, on neutral prompts.

The differential **man−woman** is built the SM-native way:

- **subspace** (pooled/hier): `remove_common([B_man, B_woman])[man]` — man's
  subspace with the man/woman *common* directions projected off (Fukui & Maki
  GDS), scored by the usual magnitude `‖B·h‖`. Directional "toward what's uniquely
  man".
- **frame**: the FRH Procrustes difference `Concept(man) − Concept(woman)`.

Directionality is read with a fixed yardstick — the **pooled** man/woman
subspaces — as `Δman`, `Δwoman` (steered − unsteered expression) per output.
One shared bf16 model serves both the frame and subspace reps. Multilingual
(8 languages, `min_lemmas=11`).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from frames.linalg import remove_common
from frames.nlp import SupportedLanguages
from frames.representations import FrameUnembeddingRepresentation
from frames.representations.concept import Concept
from subspace import ConceptSubspace, SubspaceUnembeddingRepresentation

In [ ]:
MIN_LEMMAS, MAX_TOKENS, K, STEPS, RANK = 11, 3, 4, 24, 8
LAMBDAS = [0.0, 0.4]
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
METHODS = ["pooled", "hierarchical", "frh"]
GUIDE_NAMES = ["man", "man-woman", "woman"]


def chat(u, a=""):
    return (
        f"<|start_header_id|>user<|end_header_id|>{u}<|eot_id|>"
        f"<|start_header_id|>assistant<|end_header_id|>{a}"
    )


QUESTIONS = [
    ("Tell me a short story about a doctor.", "Once"),
    ("Describe my new neighbor.", "My new neighbor"),
    ("Write about a person who just won an award.", "The winner"),
]
prompts = [chat(u, a) for u, a in QUESTIONS]
LOG = Path("resources/63_gender_steering.jsonl")
LOG.unlink(missing_ok=True)

In [ ]:
sur = SubspaceUnembeddingRepresentation.from_model_id(
    MODEL_ID, rank=RANK, r_lemma=1, language_codes=SupportedLanguages.Llama,
    torch_dtype=torch.bfloat16, device_map="auto",
)
fur = FrameUnembeddingRepresentation(model=sur.model, data=sur.data)
dev = sur.model.device


def frame_concept(name):
    c = fur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS)
    return Concept(synset=c.synset, tensor=c.tensor.to(dev))


def sub_concept(name, method):
    return sur.get_concept_cached(name, MIN_LEMMAS, MAX_TOKENS, method=method)


# --- build guides per method ------------------------------------------------
guides = {}
man_fc, woman_fc = frame_concept("man.n.01"), frame_concept("woman.n.01")
guides["frh"] = {"man": man_fc, "woman": woman_fc, "man-woman": man_fc - woman_fc}

for method in ("pooled", "hierarchical"):
    m, w = sub_concept("man.n.01", method), sub_concept("woman.n.01", method)
    stack = torch.stack([m.basis, w.basis]).to(dev)          # (2, d, k)
    diff_man = remove_common(stack)[0]                       # man minus common
    guides[method] = {
        "man": m, "woman": w,
        "man-woman": ConceptSubspace(synset="man#-woman", basis=diff_man),
    }

# fixed yardstick for directionality: the pooled man/woman subspaces
YARD_MAN, YARD_WOMAN = guides["pooled"]["man"], guides["pooled"]["woman"]
print("remove_common(man) nonzero dirs:",
      int((guides["pooled"]["man-woman"].basis.norm(dim=0) > 1e-6).sum()))

In [ ]:
def continuation(text):
    return text.split("assistant<|end_header_id|>")[-1]


base_full = sur.generate(prompts, max_new_tokens=STEPS + 2, do_sample=False)
base_cont = [continuation(t) for t in base_full]
base_man = [sur.expression(c, YARD_MAN).item() for c in base_cont]
base_woman = [sur.expression(c, YARD_WOMAN).item() for c in base_cont]

rows = []
for method in METHODS:
    for gname in GUIDE_NAMES:
        guide = guides[method][gname]
        for lam in LAMBDAS:
            torch.cuda.empty_cache()
            if method == "frh":
                texts, _ = fur.generate_with_topk_multi_guide(
                    prompts, guides=[guide], k=K, steps=STEPS, fluency_weight=lam
                )
            else:
                texts, _ = sur.generate_with_topk_guide(
                    prompts, guide=guide, k=K, steps=STEPS, fluency_weight=lam
                )
            for i, t in enumerate(texts):
                cont = continuation(t)
                dman = sur.expression(cont, YARD_MAN).item() - base_man[i]
                dwoman = sur.expression(cont, YARD_WOMAN).item() - base_woman[i]
                rows.append({
                    "method": method, "guide": gname, "lam": lam, "prompt": i,
                    "d_man": dman, "d_woman": dwoman, "direction": dman - dwoman,
                    "continuation": cont,
                })
    print(f"{method}: done")

df = pd.DataFrame(rows)
with LOG.open("w") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [ ]:
# directionality = Δman - Δwoman (positive => more male, negative => more female),
# averaged over prompts, per method x guide x lambda.
summ = (
    df.groupby(["method", "guide", "lam"])["direction"].mean().reset_index()
    .pivot_table(index=["method", "guide"], columns="lam", values="direction")
    .round(3)
)
summ.to_csv("resources/63_gender_direction_summary.csv")
print(summ.to_string())

fig, axes = plt.subplots(1, len(LAMBDAS), figsize=(6 * len(LAMBDAS), 4.2), sharey=True)
order = [(m, g) for m in METHODS for g in GUIDE_NAMES]
labels = [f"{m[:4]}·{g}" for m, g in order]
gc = {"man": "tab:blue", "man-woman": "tab:purple", "woman": "tab:red"}
for ax, lam in zip(axes, LAMBDAS):
    vals = [df[(df.method == m) & (df.guide == g) & (df.lam == lam)]["direction"].mean()
            for m, g in order]
    ax.bar(range(len(order)), vals, color=[gc[g] for _, g in order])
    ax.set_xticks(range(len(order)))
    ax.set_xticklabels(labels, rotation=60, ha="right", fontsize=8)
    ax.axhline(0, color="gray", lw=0.7)
    ax.set_title(f"λ={lam}")
    ax.set_ylabel("Δman − Δwoman  (→ male / ← female)")
plt.suptitle("Directional gender steering: man / man−woman / woman × method")
plt.tight_layout()
plt.savefig("resources/63_gender_direction.png", dpi=150)
plt.show()

In [ ]:
# qualitative gallery (λ=0)
lines = []
for i, (u, _) in enumerate(QUESTIONS):
    lines.append(f"### Prompt {i}: {u}\n\n> **baseline** — {base_cont[i].strip()}\n")
    for method in METHODS:
        for gname in GUIDE_NAMES:
            r = df[(df.method == method) & (df.guide == gname) & (df.lam == 0.0) & (df.prompt == i)].iloc[0]
            lines.append(
                f"**{method} · {gname}** (Δman {r.d_man:+.2f}, Δwoman {r.d_woman:+.2f}) — "
                f"{r.continuation.strip()}\n"
            )
gallery = "\n".join(lines)
Path("resources/63_gender_gallery.md").write_text(gallery)
print(gallery[:2500])

## Reading it

- **direction = Δman − Δwoman**: positive ⇒ output moved toward male, negative ⇒
  female (measured against the fixed pooled man/woman yardstick).
- Expected: `man` guide positive, `woman` guide negative, `man−woman` positive
  (directional) — for every method. Compare magnitudes across pooled /
  hierarchical / frame, and how λ=0.4 softens steering.
- The subspace `man−woman` is `remove_common(man)` (GDS): steer toward
  man-specific directions with the shared "person" structure removed.